<a href="https://colab.research.google.com/github/areyesp-77/intro_nlp/blob/main/session15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Librerías

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, auc, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB


In [ ]:
# Cargamos el corpus con la matriz de características
df = pd.read_csv('features_suicide.csv')

# Separamos las características (X) y los documentos (y)
X = df[['Text', 'Length', 'Verbs', 'ProperN', 'Polarity', 'Agentivity', 'Markers']]
y = df['Class']

# Cofiguramos el preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        # Vectorizar los documentos de la columna 'Text'
        ('text', TfidfVectorizer(max_features=1000), 'Text'),
        # Codificar la columna categórica 'Polarity' (neg, neu, pos)
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Polarity'])
    ],
    # Dejar intactas las columnas numéricas y binarias restantes
    remainder='passthrough'
)



In [ ]:
# Modelo base
# No olvidar que tienen tres clasificadores disponibles. Hay que probar con cuál se tiene mejor rendimiento

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    #('clasificador', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    #('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
    #('clasificador', MultinomialNB())
])

# Dividimos el corpus en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# Entrenamos el modelo
pipeline.fit(X_train, y_train)

# Evaluamos
y_pred = pipeline.predict(X_test)

print("\n--- Reporte de clasificación ---")
print(classification_report(y_test, y_pred))


In [ ]:
# Matriz de confusión detallada
cm = confusion_matrix(y_test, y_pred)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')

# Tags
group_names = [
    'Verdaderos Negativos\n(No suicidio)',
    'Falsos Positivos\n(Falsa alarma)',
    'Falsos Negativos\n(Notas omitidas)',
    'Verdaderos Positivos\n(Suicidio)'
]
group_counts = ["{0:0.0f}".format(value) for value in cm.flatten()]
group_percentages = ["{0:.2%}".format(value) for value in cm_norm.flatten()]

labels = [f"{v1}\n{v2} (Tasa: {v3})" for v1, v2, v3 in zip(group_names, group_counts, group_percentages)]
labels = np.asarray(labels).reshape(2,2)

# Graficamos la matriz de confusión
plt.figure(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=labels,
    fmt='',
    cmap='Purples',
    cbar=False,
    xticklabels=['No suicidio', 'Suicidio'],
    yticklabels=['No suicidio', 'Suicidio'],
    annot_kws={"size": 11, "weight": "bold"}
)
plt.title('Matriz de confusión', fontsize=14, pad=15)
plt.ylabel('Clase Real', fontsize=12)
plt.xlabel('Clase del modelo', fontsize=12)
plt.tight_layout()
plt.show()


## Análisis de errores

Con este tipo de análisis es posible identificar los documentos que causaron conflicto para, posteriormente, profundizar en los porqués.

Un buen análisis de errores permite generar modelos más robustos, pues la extracción de características se vuelve más fina y correlacionada con el comportamiento de los datos.


In [ ]:
# Generamos predicciones probabilísticas para evaluar la certidumbre de los errores
y_probs = pipeline.predict_proba(X_test)[:, 1]

# Creamos un DF para sistematizar
df_errors = pd.DataFrame({
    'Documento': X_test['Text'],
    'Real': y_test,
    'Modelo': y_pred,
    'Probabilidad_Riesgo': y_probs
})

# Filtramos los FN y los FP
falsos_negativos = df_errors[(df_errors['Real'] == 1) & (df_errors['Modelo'] == 0)].sort_values(by='Probabilidad_Riesgo', ascending=False)
falsos_positivos = df_errors[(df_errors['Real'] == 0) & (df_errors['Modelo'] == 1)].sort_values(by='Probabilidad_Riesgo', ascending=True)

# Resumen
print(f"=== Análisis de errores (N={len(X_test)}) ===")
print(f"Total de FN (Notas omitidas): {len(falsos_negativos)} de {y_test.sum()} reales")
print(f"Total de FP (Falsas Alarmas): {len(falsos_positivos)} de {len(y_test) - y_test.sum()} reales\n")



In [ ]:
# Ejemplos de FN
print("=== Top 10 de FN ===" + '\n')
for idx, (original_idx, row) in enumerate(falsos_negativos.head(10).iterrows(), 1):
    print(f"- {row['Documento'][:250]}..." + "\n")  #número de caracteres en la ventana



In [ ]:
# Ejemplos de FP
print("\n=== Top 10 de FP ===")
for idx, (original_idx, row) in enumerate(falsos_positivos.head(10).iterrows(), 1):
    print(f"- {row['Documento'][:250]}..." + '\n')


## Optimización del modelo

Tal como se explicó, la matriz de confusión nos aporta información específica respecto del comportamiento del modelo. Con base en ella, es posible hacer ajustes para mejorar el desempeño. Uno de esos ajustes es modificar el umbral de clasificación para hacerlo más sensitivo o no, a ciertos aspectos que se filtran a través del lenguaje.

A continuación, vamos a probar diferentes umbrales para analizar cuál de ellos nos ofrece un mejor rendimiento. Recordar: entre más bajo es el umbral, impacta más en el *recall*; entre más alto, favorece más la *precision*.

In [ ]:
# Optimzar el umbral: Modelo optimizado

# Obtenemos las probabilidades de predicción para la clase positiva (1)
y_probs = pipeline.predict_proba(X_test)[:, 1]

# Definimos el nuevo umbral (valor default: 0.5; i.e. proceso aleatorio)
umbral_custom = 0.99                    # valor de ajuste del umbral (no menor a 0.30)
y_pred_custom = (y_probs >= umbral_custom).astype(int)

# Reporte de clasificación
print(f"%%% Reporte de clasificación: umbral = {umbral_custom} %%%")
print(classification_report(y_test, y_pred_custom))

# Matriz de confusión
cm_custom = confusion_matrix(y_test, y_pred_custom)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_custom, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['No suicidio (0)', 'Suicidio (1)'],
            yticklabels=['No suicidio (0)', 'Suicidio (1)'])
plt.title(f'Matriz de Confusión - umbral {umbral_custom}')
plt.ylabel('Clase real')
plt.xlabel('Clase del Modelo')
plt.show()

## Comparación de rendimiento entre Modelo base y optimizado

Con el fin de tener elementos formales para demostrar la eficiencia del modelo, vamos a comparar cuál de los dos modelos (modelo base y optimizado) converge de forma más rápida y eficiente con una curva ROC.

In [ ]:

# Obtenemos las probabilidades de riesgo para el conjunto de test
y_probs = pipeline.predict_proba(X_test)[:, 1]

# Calculamos los FP y VP
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

# Datos del modelo base: umbral = 0.50
y_pred_base = (y_probs >= 0.50).astype(int)
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_test, y_pred_base).ravel()
tpr_050 = tp_b / (tp_b + fn_b)
fpr_050 = fp_b / (fp_b + tn_b)

# Datos del modelo optimizado: umbral = 0.78
y_pred_opt = (y_probs >= 0.78).astype(int)
tn_o, fp_o, fn_o, tp_o = confusion_matrix(y_test, y_pred_opt).ravel()
tpr_078 = tp_o / (tp_o + fn_o)
fpr_078 = fp_o / (fp_o + tn_o)

# Graficamos
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'Curva ROC (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Clasificador aleatorio')

# Destacar el punto del modelo base (umbral = 0.50)
plt.scatter(fpr_050, tpr_050, color='purple', s=200, zorder=5, edgecolor='black', marker='o',
            label=f'Modelo Base 0.50 (FPR: {fpr_050:.2%}, TPR/Recall: {tpr_050:.2%})')

# Destacar el punto del modelo optimizado (umbral = ¿?)
plt.scatter(fpr_078, tpr_078, color='orangered', s=200, zorder=5, edgecolor='black', marker='^',
            label=f'Modelo Optimizado 0.78 (FPR: {fpr_078:.2%}, TPR/Recall: {tpr_078:.2%})')

# Personalización
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('Tasa de FP', fontsize=12)
plt.ylabel('Tasa de VP', fontsize=12)
plt.title('Curva ROC: Modelo Base vs Modelo Optimizado', fontsize=14, pad=15)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.show()